# Text Classification

Building classifiers for text data using various approaches.

## Learning Objectives

- Build text classifiers with sklearn
- Use TF-IDF with traditional ML
- Implement neural text classifiers
- Evaluate classification performance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

plt.style.use('seaborn-v0_8-whitegrid')
print("Libraries loaded!")

## 1. Sample Dataset

In [ ]:
# Create sample dataset for sentiment classification
data = [
    # Positive reviews
    ("This movie was absolutely wonderful and amazing", "positive"),
    ("I loved every minute of this fantastic film", "positive"),
    ("Great acting and beautiful cinematography", "positive"),
    ("One of the best movies I have ever seen", "positive"),
    ("Highly recommend this excellent masterpiece", "positive"),
    ("The story was engaging and the characters were lovable", "positive"),
    ("A delightful and heartwarming experience", "positive"),
    ("Superb performance by the entire cast", "positive"),
    ("This film exceeded all my expectations", "positive"),
    ("A truly inspiring and uplifting movie", "positive"),
    
    # Negative reviews
    ("This movie was terrible and boring", "negative"),
    ("I hated this film completely awful", "negative"),
    ("Waste of time and money horrible movie", "negative"),
    ("Poor acting and weak storyline", "negative"),
    ("One of the worst films ever made", "negative"),
    ("Do not waste your time on this garbage", "negative"),
    ("Disappointing and frustrating to watch", "negative"),
    ("The plot made no sense at all", "negative"),
    ("Completely predictable and unoriginal", "negative"),
    ("A painful experience from start to finish", "negative"),
]

# Create DataFrame
df = pd.DataFrame(data, columns=['text', 'label'])
print(f"Dataset shape: {df.shape}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())

In [ ]:
# Split data
X = df['text'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

## 2. Naive Bayes Classifier

In [ ]:
# TF-IDF + Naive Bayes pipeline
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MultinomialNB())
])

# Train
nb_pipeline.fit(X_train, y_train)

# Predict
y_pred_nb = nb_pipeline.predict(X_test)

print("=== Naive Bayes Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_nb):.2%}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_nb))

In [ ]:
# Test on new examples
new_texts = [
    "This is an amazing and wonderful movie",
    "Terrible film, complete waste of time",
    "The movie was okay, nothing special"
]

predictions = nb_pipeline.predict(new_texts)
probabilities = nb_pipeline.predict_proba(new_texts)

print("=== New Text Predictions ===")
for text, pred, prob in zip(new_texts, predictions, probabilities):
    print(f"\nText: '{text}'")
    print(f"Prediction: {pred}")
    print(f"Confidence: {max(prob):.2%}")

## 3. Logistic Regression Classifier

In [ ]:
# TF-IDF + Logistic Regression
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000))
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

print("=== Logistic Regression Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.2%}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

In [ ]:
# Feature importance (top words)
vectorizer = lr_pipeline.named_steps['tfidf']
classifier = lr_pipeline.named_steps['clf']

feature_names = vectorizer.get_feature_names_out()
coefs = classifier.coef_[0]

# Top positive and negative words
top_positive = np.argsort(coefs)[-10:]
top_negative = np.argsort(coefs)[:10]

print("=== Top Predictive Words ===")
print("\nPositive sentiment:")
for idx in reversed(top_positive):
    print(f"  {feature_names[idx]}: {coefs[idx]:.3f}")

print("\nNegative sentiment:")
for idx in top_negative:
    print(f"  {feature_names[idx]}: {coefs[idx]:.3f}")

## 4. SVM Classifier

In [ ]:
# TF-IDF + Linear SVM
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('clf', LinearSVC())
])

svm_pipeline.fit(X_train, y_train)
y_pred_svm = svm_pipeline.predict(X_test)

print("=== SVM Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.2%}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

## 5. Compare Models

In [ ]:
# Compare all models
models = {
    'Naive Bayes': y_pred_nb,
    'Logistic Regression': y_pred_lr,
    'Linear SVM': y_pred_svm
}

print("=== Model Comparison ===")
print(f"{'Model':<25} {'Accuracy':>10}")
print("-" * 37)
for name, preds in models.items():
    acc = accuracy_score(y_test, preds)
    print(f"{name:<25} {acc:>10.2%}")

In [ ]:
# Visualize comparison
accuracies = [accuracy_score(y_test, preds) for preds in models.values()]

plt.figure(figsize=(10, 5))
bars = plt.bar(models.keys(), accuracies, color=['steelblue', 'coral', 'seagreen'])
plt.ylabel('Accuracy')
plt.title('Model Performance Comparison')
plt.ylim(0, 1)

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{acc:.1%}', ha='center', fontsize=12)

plt.tight_layout()
plt.show()

## 6. Neural Network Classifier

In [ ]:
# Prepare data for neural network
from collections import Counter

# Build vocabulary
def build_vocab(texts, min_freq=1):
    word_counts = Counter()
    for text in texts:
        words = text.lower().split()
        word_counts.update(words)
    
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, count in word_counts.items():
        if count >= min_freq:
            vocab[word] = len(vocab)
    return vocab

def encode_text(text, vocab, max_len=20):
    words = text.lower().split()
    indices = [vocab.get(w, vocab['<UNK>']) for w in words]
    
    # Pad or truncate
    if len(indices) < max_len:
        indices += [vocab['<PAD>']] * (max_len - len(indices))
    else:
        indices = indices[:max_len]
    return indices

vocab = build_vocab(X_train)
print(f"Vocabulary size: {len(vocab)}")

In [ ]:
# Dataset class
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=20):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len
        self.label_map = {'positive': 1, 'negative': 0}
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.label_map[self.labels[idx]]
        
        encoded = encode_text(text, self.vocab, self.max_len)
        return torch.tensor(encoded), torch.tensor(label, dtype=torch.float)

# Create datasets
train_dataset = TextDataset(X_train, y_train, vocab)
test_dataset = TextDataset(X_test, y_test, vocab)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# Neural classifier model
class NeuralTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.embedding(x)  # (batch, seq_len, embed_dim)
        pooled = embedded.mean(dim=1)  # (batch, embed_dim)
        hidden = torch.relu(self.fc1(pooled))
        hidden = self.dropout(hidden)
        output = self.fc2(hidden).squeeze(1)
        return output

# Create model
model = NeuralTextClassifier(
    vocab_size=len(vocab),
    embed_dim=50,
    hidden_dim=32
)
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# Training
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss()

n_epochs = 50
train_losses = []

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    
    for texts, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    train_losses.append(epoch_loss / len(train_loader))
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {train_losses[-1]:.4f}")

In [ ]:
# Evaluate
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for texts, labels in test_loader:
        outputs = model(texts)
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()
        
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

# Convert to labels
pred_labels = ['positive' if p == 1 else 'negative' for p in all_preds]
true_labels = ['positive' if l == 1 else 'negative' for l in all_labels]

print("=== Neural Network Results ===")
print(f"Accuracy: {accuracy_score(true_labels, pred_labels):.2%}")
print(f"\nClassification Report:")
print(classification_report(true_labels, pred_labels))

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(train_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Neural Network Training Loss')
plt.tight_layout()
plt.show()

## 7. Multi-Class Classification

In [ ]:
# Multi-class dataset
multi_data = [
    ("Stock market shows strong gains today", "finance"),
    ("Investors worried about inflation rates", "finance"),
    ("New IPO breaks trading records", "finance"),
    
    ("Team wins championship game", "sports"),
    ("Player scores winning goal", "sports"),
    ("Coach announces new strategy", "sports"),
    
    ("New smartphone features announced", "tech"),
    ("AI breakthrough in machine learning", "tech"),
    ("Software update improves performance", "tech"),
]

multi_df = pd.DataFrame(multi_data, columns=['text', 'category'])
print(f"Categories: {multi_df['category'].unique()}")

In [ ]:
# Multi-class pipeline
X_multi = multi_df['text'].values
y_multi = multi_df['category'].values

multi_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(multi_class='multinomial', max_iter=1000))
])

multi_pipeline.fit(X_multi, y_multi)

# Test predictions
test_texts = [
    "Bitcoin price reaches new high",
    "New gaming console released",
    "Basketball team makes playoffs"
]

predictions = multi_pipeline.predict(test_texts)
probabilities = multi_pipeline.predict_proba(test_texts)

print("=== Multi-Class Predictions ===")
for text, pred, probs in zip(test_texts, predictions, probabilities):
    print(f"\nText: '{text}'")
    print(f"Prediction: {pred}")
    for cat, prob in zip(multi_pipeline.classes_, probs):
        print(f"  {cat}: {prob:.1%}")

## 8. Confusion Matrix Visualization

In [ ]:
# Confusion matrix for binary classification
cm = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap='Blues')
plt.colorbar()

classes = ['negative', 'positive']
plt.xticks([0, 1], classes)
plt.yticks([0, 1], classes)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')

# Add values
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=20)

plt.tight_layout()
plt.show()

## 9. Key Takeaways

### Algorithm Comparison

| Model | Pros | Cons |
|-------|------|------|
| **Naive Bayes** | Fast, works well with small data | Strong independence assumption |
| **Logistic Reg** | Interpretable, probabilistic | Linear decision boundary |
| **SVM** | Good margins, effective | Less interpretable |
| **Neural Net** | Learns features | Needs more data |

### Best Practices

1. Start with simple models (Naive Bayes, Logistic Regression)
2. Use TF-IDF or n-grams for feature extraction
3. Handle class imbalance with stratified splits
4. Evaluate with appropriate metrics (precision, recall, F1)
5. Use cross-validation for robust evaluation

### Next Steps
- Transformers for state-of-the-art classification
- Fine-tuning pre-trained models (BERT)